## Exercise — Curved relationships & polynomial features

### Task 1 — Does the straight line miss a curve?

A new series of 30 antibacterial compounds has activity (pIC50) that depends on lipophilicity (logP). Start your script with this setup:

```py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

np.random.seed(0)
logp = np.linspace(-1, 5, 30)
noise = np.random.normal(0, 0.25, 30)
pic50 = 3.5 + 1.0 * logp - 0.2 * logp ** 2 + noise
df = pd.DataFrame({"logp": logp, "pic50": pic50})
```

Then do the following:

1. Write the straight-line model `linear(logp, slope, intercept)` from the video.
2. Fit it with `curve_fit` and unpack `slope` and `intercept`.
3. Compute the predictions `pred_line` and the residuals `resid_line` (observed minus predicted).

**Before you run anything**, write down your prediction: will the residual be positive or negative for the first compound (logP = −1), for compound 15 (logP ≈ 2.1) and for the last compound (logP = 5)?

Next, print those three residuals. Use `.iloc[0]`, `.iloc[15]` and `.iloc[-1]`, and round each to 2 decimals.

Finally, draw the residuals against logP with a grey line at zero. Label both axes with units and give the plot a title. Check whether the plot shows an arch.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

np.random.seed(0)
logp = np.linspace(-1, 5, 30)
noise = np.random.normal(0, 0.25, 30)
pic50 = 3.5 + 1.0 * logp - 0.2 * logp ** 2 + noise
df = pd.DataFrame({"logp": logp, "pic50": pic50})

def linear(logp, slope, intercept):
    return slope * logp + intercept

popt, pcov = curve_fit(linear, df["logp"], df["pic50"])
slope, intercept = popt
pred_line = linear(df["logp"], slope, intercept)
resid_line = df["pic50"] - pred_line
print("first:", round(resid_line.iloc[0], 2))
print("middle:", round(resid_line.iloc[15], 2))
print("last:", round(resid_line.iloc[-1], 2))
plt.figure(figsize=(8, 5))
plt.scatter(df["logp"], resid_line)
plt.plot([-1, 5], [0, 0], color="grey")
plt.xlabel("logP (unitless)")
plt.ylabel("Residual (pIC50 units)")
plt.title("Residuals of the straight-line fit")
plt.grid(True)
plt.show()

### Task 2 — Add a squared term

Write the model `quadratic(logp, a, b, c)`, which returns $a\,x^2 + b\,x + c$. Fit it to `df` with `curve_fit` and unpack `a`, `b` and `c`. Print each of them rounded to 3 decimals. Then print the optimal logP, $x_{opt} = -b / 2a$, rounded to 2 decimals.

Next, consider the code below. It treats logP² as an ordinary precomputed column, so the model is a plain weighted sum of two columns:

```py
df["logp_sq"] = df["logp"] ** 2

def quad_columns(df, a, b, c):
    return a * df["logp_sq"] + b * df["logp"] + c
```

**Predict:** if you fit `quad_columns` to `df` and `df["pic50"]`, will its `a`, `b` and `c` differ from the ones you just got? Run the code, fit it, and print its three coefficients rounded to 3 decimals to check your prediction.


In [ ]:
def quadratic(logp, a, b, c):
    return a * logp ** 2 + b * logp + c

popt, pcov = curve_fit(quadratic, df["logp"], df["pic50"])
a, b, c = popt
print("a:", round(a, 3), "b:", round(b, 3), "c:", round(c, 3))
print("optimal logP:", round(-b / (2 * a), 2))
df["logp_sq"] = df["logp"] ** 2

def quad_columns(df, a, b, c):
    return a * df["logp_sq"] + b * df["logp"] + c

popt, pcov = curve_fit(quad_columns, df, df["pic50"])
a2, b2, c2 = popt
print("column model a:", round(a2, 3), "b:", round(b2, 3), "c:", round(c2, 3))

### Task 3 — An interaction between logP and donors

Add a hydrogen-bond donor count (0 to 3) and a simulated log permeability (log cm/s) to the table:

```py
df["hbd"] = np.arange(30) % 4
perm_noise = np.random.normal(0, 0.1, 30)
df["log_perm"] = -5.5 + 0.6 * df["logp"] - 0.15 * df["hbd"] - 0.12 * df["logp"] * df["hbd"] + perm_noise
```

Write the model `interaction(df, k_logp, k_hbd, k_both, intercept)` from the video. Fit it to `df` with `df["log_perm"]` as the target, and unpack the four coefficients.

Print the interaction coefficient `k_both`, rounded to 3 decimals. Then print the logP slope, $k_1 + k_3\,\text{HBD}$, for 0, 1, 2 and 3 donors, each rounded to 3 decimals. Use the numbers to check how each extra donor changes the effect of logP.


In [ ]:
df["hbd"] = np.arange(30) % 4
perm_noise = np.random.normal(0, 0.1, 30)
df["log_perm"] = -5.5 + 0.6 * df["logp"] - 0.15 * df["hbd"] - 0.12 * df["logp"] * df["hbd"] + perm_noise

def interaction(df, k_logp, k_hbd, k_both, intercept):
    return k_logp * df["logp"] + k_hbd * df["hbd"] + k_both * df["logp"] * df["hbd"] + intercept

popt, pcov = curve_fit(interaction, df, df["log_perm"])
k_logp, k_hbd, k_both, k0 = popt
print("interaction coefficient:", round(k_both, 3))
for n_donors in range(4):
    print(n_donors, "donors: logP slope =", round(k_logp + k_both * n_donors, 3))

### Task 4 — Fix the jagged curve

A colleague drew the quadratic fit from Task 2 over the activity data with this code:

```py
logp_grid = np.linspace(-1, 5, 5)
plt.figure(figsize=(8, 5))
plt.scatter(df["logp"], df["pic50"], label="compounds")
plt.plot(logp_grid, quadratic(logp_grid, a, b, c), label="quadratic")
plt.xlabel("logP (unitless)")
plt.ylabel("pIC50 (-log10 M)")
plt.title("Activity against lipophilicity")
plt.legend()
plt.grid(True)
plt.show()
```

The "curve" comes out as a few straight segments with sharp corners, and it cuts off the top of the peak. Correct the line that causes this, so the curve is drawn smoothly on 200 logP values. Then run the whole figure.


In [ ]:
logp_grid = np.linspace(-1, 5, 200)
plt.figure(figsize=(8, 5))
plt.scatter(df["logp"], df["pic50"], label="compounds")
plt.plot(logp_grid, quadratic(logp_grid, a, b, c), label="quadratic")
plt.xlabel("logP (unitless)")
plt.ylabel("pIC50 (-log10 M)")
plt.title("Activity against lipophilicity")
plt.legend()
plt.grid(True)
plt.show()

### Task 5 — Is the bend really gone?

Compute `pred_quad` and `resid_quad` for the quadratic fit from Task 2.

**Before running**, predict whether the quadratic's training R² can come out *lower* than the straight line's. Then print both R² values with `r_squared`, rounded to 3 decimals, to check your prediction.

A higher training score alone proves nothing, so finish with the real test. Draw `resid_quad` against logP with a grey line at zero, using the same labels as in Task 1 and the title "Residuals of the quadratic fit". Check that the residuals now scatter evenly around zero, with no arch.


In [ ]:
pred_quad = quadratic(df["logp"], a, b, c)
resid_quad = df["pic50"] - pred_quad
print("R2 straight line:", round(r_squared(df["pic50"], pred_line), 3))
print("R2 quadratic:   ", round(r_squared(df["pic50"], pred_quad), 3))
plt.figure(figsize=(8, 5))
plt.scatter(df["logp"], resid_quad)
plt.plot([-1, 5], [0, 0], color="grey")
plt.xlabel("logP (unitless)")
plt.ylabel("Residual (pIC50 units)")
plt.title("Residuals of the quadratic fit")
plt.grid(True)
plt.show()